
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# LAB- Evaluation with Mosaic AI Agent Evaluation

In this lab, you will have the opportunity to evaluate a RAG chain model **using Mosaic AI Agent Evaluation Framework.**

**Lab Outline:**

*In this lab, you will complete the following tasks:*

- **Task 1**: Define a custom Gen AI evaluation metric.

- **Task 2**: Conduct an evaluation test using the Agent Evaluation Framework.

- **Task 3**: Analyze the evaluation results through the user interface.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**



## Classroom Setup

Install required libraries.

In [0]:
%pip install -U -qq databricks-agents databricks-sdk databricks-vectorsearch databricks-langchain langchain==0.3.7 langchain-community==0.3.7 mlflow>=3.0 databricks-feature-engineering --upgrade

In [0]:
dbutils.library.restartPython()

Before starting the lab, run the provided classroom setup script. This script will define configuration variables necessary for the lab. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-04

**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Lab Overview




## Evaluation Dataset

In this lab, you will work with the same dataset utilized in the demos. This dataset contains sample queries along with their corresponding expected responses, which are generated using synthetic data.

In [0]:
display(DA.eval_df)

## Load the Model

A RAG chain has been created and registered for use in this lab. The model details are provided below.

**📌 Note:** If you are using your own workspace to run this lab, you must manually execute **`00 - Build Model / 00-Build Model`**.

In [0]:
import mlflow

catalog_name = "genai_shared_catalog_03"
schema_name = f"ws_{spark.conf.get('spark.databricks.clusterUsageTags.clusterOwnerOrgId')}"

mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{catalog_name}.{schema_name}.rag_app/1"
model_name = f"{catalog_name}.{schema_name}.rag_app"

## Task 1 - Define A Custom Metric

For this task, define a custom metric to evaluate whether the generated **"ANSWER"** from the RAG chain is easily readable by a non-expert user.

In [0]:
from mlflow.metrics.genai import make_genai_metric_from_prompt

## Prompt for LLM as judge to determine if the generated response is easily readable by non-academic or expert readers
eval_prompt = "Your task is to determine whether the generated response easily readable by non-academic or expert readers. This was the content: '{retrieved_context}'"

## Use Llama-3 as LLM
llm="endpoints:/databricks-meta-llama-3-3-70b-instruct"

## Define the metric
is_readable = make_genai_metric_from_prompt(
    name="is_readable",
    judge_prompt=eval_prompt,
    model=llm,
    metric_metadata={"assessment_type": "ANSWER"},
)

##Task 2 - Run Evaluation Test

Next, run an evaluation using the custom metric you defined. Ensure that you select **Mosaic AI Agent Evaluation** as the evaluation type.


In [0]:
with mlflow.start_run(run_name="lab_04_agent_evaluation"):
    eval_results = mlflow.evaluate(
        data = DA.eval_df,
        model = model_uri,
        model_type = "databricks-agent",
        extra_metrics=[is_readable]
    )

## Task 3 - Review Evaluation Results

Review the evaluation results in the **Experiments** section. Examine the following information regarding this evaluation:

- Token usage

- Model metrics

- Results of the custom metric defined earlier ("readability")

## Conclusion

In this lab, you evaluated a RAG chain using the Mosaic AI Evaluation Framework library. You began by loading the dataset and RAG model. Then, you defined a custom metric and initiated the evaluation process. Finally, you reviewed the results through the user interface.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>